In [33]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils

from matplotlib import pyplot as plt
from scipy.spatial.distance import mahalanobis


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [34]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_distances_suffixes.parquet"))

In [35]:
# Council district dummies + some cleaning

vals = ['1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','CITYWIDE']
for v in vals:
    df[f'cd_{v}'] = 0

invalids = []
for idx, row in df.iterrows():
    splitted = re.split(r"[,;]", row['council_district'])
    council_districts = [x.strip() for x in splitted if len(x.strip())>0]
    for cd in council_districts:
        if cd in vals:
            df.loc[idx, f'cd_{cd}'] = 1
        else:
            invalids.append((idx, cd))

print(invalids)

# apply fixes
df.loc[82, "cd_9"] = 1
df.loc[571, "cd_CITYWIDE"] = 1
df.loc[797, "cd_CITYWIDE"] = 1
df.loc[808, "cd_CITYWIDE"] = 1

[(82, 'Multiple'), (571, 'ALL'), (797, 'ALL'), (808, 'ALL')]


In [36]:
# Year dummies

for y in range(2018, 2027):
    df[f'yr_{y}'] = 0
    df.loc[df['year'].astype('int')==y, f'yr_{y}'] = 1


In [37]:
# processed variables

# Outcome variable
df["outcome"] = 1
df.loc[df["project_result"] == "APPROVED", "outcome"] = 2
df.loc[df["project_result"].isin(['DELAYED', 'DENIED', 'APPLICATION WIDTHDRAWN']), "outcome"] = 0

# Choose distance measure
df["atypicality"] = df["mahalanobis"]

# project type dummies
df['is_residential'] = df['project_type'].isin(['RESIDENTIAL DEVELOPMENT', 'PERMANENT SUPPORTIVE HOUSING / HOMELESS SHELTER', 'SENIOR CARE / ASSISTED LIVING FACILITY'])
df['is_mixed_use'] = df['project_type'].isin(['MIXED-USE DEVELOPMENT'])
df['is_nonresidential'] = df['project_type'].isin(['NON-RESIDENTIAL DEVELOPMENT'])

# support/opposition letters
df["log2_support"] = np.log2(df["n_support"] + 1)
df["log2_oppose"] = np.log2(df["n_oppose"] + 1)

# height/square footage
df["height"] = pd.to_numeric(df["height"], errors="coerce")
df["square_footage"] = pd.to_numeric(df["square_footage"], errors="coerce")
df["log_square_footage"] = np.log(1 + df["square_footage"])

# handle missing values for height and log_square_footage
df["height_missing"] = df["height"].isna()
df["height"] = df["height"].fillna(0)
df["log_square_footage_missing"] = df["log_square_footage"].isna()
df["log_square_footage"] = df["log_square_footage"].fillna(0)





In [38]:
# Output regression data

df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))

In [39]:
# Running R regressions

res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/15-ordered-logit.R"], check=True, capture_output=True, text=True)
print(res.stdout)


                                    Dependent variable:         
                           -------------------------------------
                                          outcome               
                             (1)      (2)       (3)       (4)   
----------------------------------------------------------------
is_residential              0.531    0.455     0.558     0.195  
                           (0.327)  (0.335)   (0.346)   (0.361) 
                                                                
is_mixed_use                0.157    0.093     0.152    -0.285  
                           (0.326)  (0.335)   (0.346)   (0.368) 
                                                                
is_nonresidential           -0.310   -0.414   -0.326    -0.346  
                           (0.323)  (0.330)   (0.338)   (0.343) 
                                                                
log_square_footage          -0.035   -0.025   -0.015    -0.053  
                        

In [40]:
# Load regression coefficients

coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_coefs.parquet"))


In [41]:
# Coefficient results

MAIN_REG = 'r4'

AtypicalityCoef = coefs_df.loc[
    (coefs_df['regression_name']==MAIN_REG) & (coefs_df['coef_name']=='atypicality'),
    'estimate'
].values[0]

NumOpposeCoef = coefs_df.loc[
    (coefs_df['regression_name']==MAIN_REG) & (coefs_df['coef_name']=='log2_oppose'),
    'estimate'
].values[0]

ConCalCoef = coefs_df.loc[
    (coefs_df['regression_name']==MAIN_REG) & (coefs_df['coef_name']=='is_consent_calendarTRUE'),
    'estimate'
].values[0]

AgendaOrderCoef = coefs_df.loc[
    (coefs_df['regression_name']==MAIN_REG) & (coefs_df['coef_name']=='agenda_order'),
    'estimate'
].values[0]

RESULTS = {
    "AtypicalityCoef": f"{np.abs(AtypicalityCoef):.3f}",
    "NumOpposeCoef": f"{np.abs(NumOpposeCoef):.3f}",
    "ConCalCoef": f"{np.abs(ConCalCoef):.3f}",
    "AgendaOrderCoef": f"{np.abs(AgendaOrderCoef):.3f}",
}

_ = wt.update_results(RESULTS)

RESULTS

{'AtypicalityCoef': '0.206',
 'NumOpposeCoef': '0.160',
 'ConCalCoef': '1.708',
 'AgendaOrderCoef': '0.136'}

In [42]:
# Output table

header = r"""\begin{table}[H]
\centering
\caption{Factors Influencing Approval and Delay}
\vspace{0.2cm}
\label{tab_ologit_results}
\begin{adjustbox}{max height=0.45\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\multicolumn{5}{c}{Ordered Logit Model} \\
\multicolumn{5}{c}{0=Delayed/Denied, 1=Approved with Conditions, 2=Approved} \\
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports coefficient estimates from the ordered logit regression described in Section \ref{sec_methodology}. Dummies for missing values of height and square footage are included for cases without reported physical dimensions.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
reg_names = ["r1", "r2", "r3","r4"]

vars = [
    ("is_residentialTRUE", "Residential Development"),
    ("is_mixed_useTRUE", "Mixed-Use Development"),
    ("is_nonresidentialTRUE", "Non-Residential Development"),
    ("log_square_footage", "$\\ln$(Square Footage)"),
    ("height", "Height (ft)"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)"),
    ("agenda_order", "Agenda Order"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendarTRUE", "Consent Calendar"),
    ("atypicality", "Atypicality"),
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += "\n & & & & \\\\ \n"

tbl += "Suffix Group Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="sfx_grp_CUPTRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Council District Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cd_1")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Year Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="yr_2019")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Embedding Cluster Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cluster_fe1TRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"



tbl += "\n & & & & \\\\ \n"

vars = [
    ("0|1", "$\\mu_0$"),
    ("1|2", "$\\mu_1$")
]

for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

tbl += "McFadden's Pseudo-R$^2$"
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="pseudo_r2")
    if idx.sum()==0:
        tbl += " & "
    else:
        tbl += f" & {coefs_df.loc[idx, "estimate"].values[0]:.3f}"
tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_ologit_results.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)


\begin{table}[H]
\centering
\caption{Factors Influencing Approval and Delay}
\vspace{0.2cm}
\label{tab_ologit_results}
\begin{adjustbox}{max height=0.45\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\multicolumn{5}{c}{Ordered Logit Model} \\
\multicolumn{5}{c}{0=Delayed/Denied, 1=Approved with Conditions, 2=Approved} \\
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  &  \\
Residential Development  & 0.531$^{}$ & 0.455$^{}$ & 0.558$^{}$ & 0.195$^{}$ \\
 & (0.327) & (0.335) & (0.346) & (0.361) \\ [1.8ex]
Mixed-Use Development  & 0.157$^{}$ & 0.093$^{}$ & 0.152$^{}$ & -0.285$^{}$ \\
 & (0.326) & (0.335) & (0.346) & (0.368) \\ [1.8ex]
Non-Residential Development  & -0.310$^{}$ & -0.414$^{}$ & -0.326$^{}$ & -0.346$^{}$ \\
 & (0.323) & (0.330) & (0.338) & (0.343) \\ [1.8ex]
$\ln$(Square Footage)  & -0.035$^{}$ & -0.025$^{}$ & -0.015$^{}$ & -0.053$^{}$ \\
 & (0.059) & (0.061) & (0.064) & (0.064) \\ [1.8ex]
Height (ft)  & -0.001$^{}$ & -0.001$^{}$ & -0.001$^{}$ & -0.001$^{}

In [43]:
# Load marginals

marginals_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_marginals.parquet"))

In [44]:
# Marginal effects results

_results = {}
for term in ["atypicality", "log2_oppose", "is_consent_calendar", "agenda_order"]:
    for group in ["0", "1", "2"]:
        mask = (marginals_df['regression_name']==MAIN_REG) & (marginals_df['term']==term) & (marginals_df['group']==group)
        _results[(term, group)] = marginals_df.loc[mask, 'estimate'].values[0]

RESULTS = {
    "AtypicalityME2": f"{np.abs(_results[('atypicality', '2')])*100:.1f}",
    "AtypicalityME1": f"{np.abs(_results[('atypicality', '1')])*100:.1f}",
    "AtypicalityME0": f"{np.abs(_results[('atypicality', '0')])*100:.1f}",
    "NumOpposeME2": f"{np.abs(_results[('log2_oppose', '2')])*100:.1f}",
    "NumOpposeME1": f"{np.abs(_results[('log2_oppose', '1')])*100:.1f}",
    "NumOpposeME0": f"{np.abs(_results[('log2_oppose', '0')])*100:.1f}",
    "ConCalME2": f"{np.abs(_results[('is_consent_calendar', '2')])*100:.1f}",
    "ConCalME1": f"{np.abs(_results[('is_consent_calendar', '1')])*100:.1f}",
    "ConCalME0": f"{np.abs(_results[('is_consent_calendar', '0')])*100:.1f}",
    "AgendaOrderME2": f"{np.abs(_results[('agenda_order', '2')])*100:.1f}",
    "AgendaOrderME1": f"{np.abs(_results[('agenda_order', '1')])*100:.1f}",
    "AgendaOrderME0": f"{np.abs(_results[('agenda_order', '0')])*100:.1f}",
}

_ = wt.update_results(RESULTS)

RESULTS

{'AtypicalityME2': '4.2',
 'AtypicalityME1': '1.8',
 'AtypicalityME0': '2.4',
 'NumOpposeME2': '3.3',
 'NumOpposeME1': '1.4',
 'NumOpposeME0': '1.9',
 'ConCalME2': '34.9',
 'ConCalME1': '20.7',
 'ConCalME0': '14.2',
 'AgendaOrderME2': '2.8',
 'AgendaOrderME1': '1.2',
 'AgendaOrderME0': '1.6'}

In [45]:
# Marginal effects table

header = r"""\begin{table}[H]
\centering
\caption{Factors Influencing Approval and Delay - Marginal Effects}
\vspace{0.2cm}
\label{tab_ologit_marginal_effects}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lccc}
\toprule
 & (1) & (2) & (3) \\
\midrule
 & \makecell{Delay /\\Denied} & \makecell{Approved with\\Conditions} & \makecell{Approved} \\
 &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports estimated marginal effects from the ordered logit regression in column (4) of Table \ref{tab_ologit_results}.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
outcomes = ["0", "1", "2"]

vars = [
    ("is_residential", "Residential Development"),
    ("is_mixed_use", "Mixed-Use Development"),
    ("is_nonresidential", "Non-Residential Development"),
    ("log_square_footage", "$\\ln$(Square Footage)"),
    ("height", "Height (ft)"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)"),
    ("agenda_order", "Agenda Order"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendar", "Consent Calendar"),
    ("atypicality", "Atypicality"),
]

tbl = ""
for v in vars:  # use vars from previous cell
    tbl += v[1] + " "
    for y in outcomes:
        idx = (marginals_df["regression_name"]==MAIN_REG) & (marginals_df["term"]==v[0]) & (marginals_df["group"]==y)
        if idx.sum()==0:
            tbl += " & "
            continue
        meff = marginals_df.loc[idx, "estimate"].values[0]
        serr = marginals_df.loc[idx, "std.error"].values[0]
        stars = utils.stars(meff, serr)
        tbl += f" & {meff:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for y in outcomes:
        idx = (marginals_df["regression_name"]==MAIN_REG) & (marginals_df["term"]==v[0]) & (marginals_df["group"]==y)
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = marginals_df.loc[idx, "std.error"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_ologit_marginal_effects.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)


\begin{table}[H]
\centering
\caption{Factors Influencing Approval and Delay - Marginal Effects}
\vspace{0.2cm}
\label{tab_ologit_marginal_effects}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lccc}
\toprule
 & (1) & (2) & (3) \\
\midrule
 & \makecell{Delay /\\Denied} & \makecell{Approved with\\Conditions} & \makecell{Approved} \\
 &  &  &  \\
Residential Development  & -0.023$^{}$ & -0.018$^{}$ & 0.040$^{}$ \\
 & (0.041) & (0.034) & (0.075) \\ [1.8ex]
Mixed-Use Development  & 0.035$^{}$ & 0.023$^{}$ & -0.058$^{}$ \\
 & (0.046) & (0.028) & (0.074) \\ [1.8ex]
Non-Residential Development  & 0.044$^{}$ & 0.027$^{}$ & -0.071$^{}$ \\
 & (0.046) & (0.023) & (0.069) \\ [1.8ex]
$\ln$(Square Footage)  & 0.006$^{}$ & 0.005$^{}$ & -0.011$^{}$ \\
 & (0.008) & (0.006) & (0.013) \\ [1.8ex]
Height (ft)  & 0.000$^{}$ & 0.000$^{}$ & -0.000$^{}$ \\
 & (0.000) & (0.000) & (0.000) \\ [1.8ex]
$\log_2$(\# Support)  & -0.011$^{}$ & -0.008$^{}$ & 0.018$^{}$ \\
 & (0.007) & (0.

In [46]:
# Oster (2019) delta*

oster_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_oster_delta.parquet"))
delta_star = oster_df.loc[0, "Value"]

RESULTS = {
    "OsterDelta": f"{delta_star:.2f}"
}

_ = wt.update_results(RESULTS)

RESULTS

{'OsterDelta': '2.76'}

In [47]:
# Load regression coefficients

coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_coefs_nodelay.parquet"))


In [48]:
# Output table

header = r"""\begin{table}[H]
\centering
\caption{Factors Influencing Approval with Conditions - Non-Delay Outcomes}
\vspace{0.2cm}
\label{tab_blogit_results}
\begin{adjustbox}{max height=0.45\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\multicolumn{5}{l}{Binary Logit Model} \\
\multicolumn{5}{l}{0=Approved with Conditions/Denied, 1=Approved} \\
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  & \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports coefficient estimates from the binary logit regression described in Section \ref{sec_results}, in which cases where delay was the outcome were removed from the estimation sample. The outcome is 1 if the case is approved, and 0 if the case is approved with conditions or denied. Dummies for missing values of height and square footage are included for cases without reported physical dimensions.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
reg_names = ["r1b", "r2b", "r3b","r4b"]

vars = [
    ("is_residentialTRUE", "Residential Development"),
    ("is_mixed_useTRUE", "Mixed-Use Development"),
    ("is_nonresidentialTRUE", "Non-Residential Development"),
    ("log_square_footage", "$\\ln$(Square Footage)"),
    ("height", "Height (ft)"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)"),
    ("agenda_order", "Agenda Order"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendarTRUE", "Consent Calendar"),
    ("atypicality", "Atypicality"),
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += "\n & & & & \\\\ \n"

tbl += "Suffix Group Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="sfx_grp_CUPTRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Council District Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cd_1")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Year Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="yr_2019")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Embedding Cluster Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cluster_fe1TRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "\n & & & & \\\\ \n"


tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

tbl += "McFadden's Pseudo-R$^2$"
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="pseudo_r2")
    if idx.sum()==0:
        tbl += " & "
    else:
        tbl += f" & {coefs_df.loc[idx, "estimate"].values[0]:.3f}"
tbl += r" \\ [1.8ex]" + "\n"



table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_blogit_results.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)


\begin{table}[H]
\centering
\caption{Factors Influencing Approval with Conditions - Non-Delay Outcomes}
\vspace{0.2cm}
\label{tab_blogit_results}
\begin{adjustbox}{max height=0.45\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\multicolumn{5}{l}{Binary Logit Model} \\
\multicolumn{5}{l}{0=Approved with Conditions/Denied, 1=Approved} \\
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  & \\
Residential Development  & 0.202$^{}$ & 0.225$^{}$ & 0.470$^{}$ & 0.188$^{}$ \\
 & (0.380) & (0.391) & (0.413) & (0.436) \\ [1.8ex]
Mixed-Use Development  & 0.040$^{}$ & 0.071$^{}$ & 0.295$^{}$ & -0.053$^{}$ \\
 & (0.383) & (0.398) & (0.420) & (0.452) \\ [1.8ex]
Non-Residential Development  & -0.628$^{}$ & -0.672$^{*}$ & -0.543$^{}$ & -0.572$^{}$ \\
 & (0.400) & (0.408) & (0.428) & (0.440) \\ [1.8ex]
$\ln$(Square Footage)  & -0.037$^{}$ & -0.033$^{}$ & -0.060$^{}$ & -0.089$^{}$ \\
 & (0.073) & (0.076) & (0.080) & (0.081) \\ [1.8ex]
Height (ft)  & -0.001$^{}$ & -0.001$^{}$ & -0.001$^{